# RDD2022 India — quick Roboflow v5 run
Run one cell at a time. Checkpoints are saved to Drive after every epoch.


## 1. Check T4 GPU first


In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
assert torch.cuda.is_available(), "Set Runtime > Change runtime type > T4 GPU first."


## 2. Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


## 3. Install tools and upload only Road_Damage_Roboflow_Quick.zip (not India.zip)


In [ ]:
!pip -q install roboflow ultralytics gradio pyyaml
from google.colab import files
_ = files.upload()


In [ ]:
from pathlib import Path
import zipfile
p=Path("/content/Road_Damage_Roboflow_Quick.zip")
assert p.exists(), "Upload Road_Damage_Roboflow_Quick.zip first."
with zipfile.ZipFile(p) as z: z.extractall("/content")
print("Project ready:", Path("/content/road_damage_github/src/prepare_roboflow.py").exists())


## 4. Download the published v5 YOLOv8 export directly into Colab
You need a Roboflow API key with Universe access. The input below is hidden. The public dataset contains 7,706 images; the next cell keeps a small subset.


In [ ]:
from getpass import getpass
from roboflow import Roboflow
key = getpass("Roboflow API key (hidden): ")
rf = Roboflow(api_key=key)
dataset = rf.workspace("prakhar-kpb1v").project("rdd2022-india-il8ju").version(5).download("yolov8", location="/content/rdd_rf")
del key
print("Export folder:", dataset.location)


## 5. Keep 600 train, 100 validation, 100 test images in four classes, then reclaim disk space


In [ ]:
%cd /content/road_damage_github
!python src/prepare_roboflow.py --source /content/rdd_rf --out data/roboflow_small --train 600 --val 100 --test 100
import shutil
shutil.rmtree("/content/rdd_rf")
print("Full temporary export removed; small subset retained.")


## 6. Train 8 epochs on the T4 and evaluate on the held-out test
The Drive folder is `MyDrive/RoadDamageResults/mini_india/weights/`. Run this cell once and wait; do not change runtimes during training.


In [ ]:
!python src/train.py --data data/roboflow_small/data.yaml --epochs 8 --imgsz 512 --batch 16 --device 0 --project /content/drive/MyDrive/RoadDamageResults


## 7. Package and download the GitHub submission


In [ ]:
from pathlib import Path
import shutil
from google.colab import files
root=Path("/content/road_damage_github")
w=Path("/content/drive/MyDrive/RoadDamageResults/mini_india/weights/best.pt")
assert w.exists(), "No checkpoint yet; do not claim a trained model."
(root/"models").mkdir(exist_ok=True)
shutil.copy2(w, root/"models/best.pt")
stage=Path("/content/submission/road_damage_github")
if stage.parent.exists(): shutil.rmtree(stage.parent)
shutil.copytree(root, stage, ignore=shutil.ignore_patterns("data", "__pycache__", "*.zip", "*.pyc"))
zip_path=Path(shutil.make_archive("/content/Road_Damage_GitHub_Submission", "zip", stage.parent))
print("Final ZIP:", zip_path, "size MB:", round(zip_path.stat().st_size/1e6, 1))
files.download(str(zip_path))


## 8. GitHub and form
Extract the final ZIP. Create a public GitHub repository. Upload the files **inside** `road_damage_github` (not the outer ZIP). Submit its URL to the Google Form by 8 PM IST. If time is tight, upload the starter project now and add weights/results later to the same repository. Never expose your Roboflow key or raw images.
